In [32]:
import pandas as pd

In [37]:
df = pd.read_csv('crop_medicine_uses_dataset.csv')

In [34]:
df

,Crop_Medicine,Uses_When_to_Use
0,Neem Oil,"For rice crops, use when aphids is seen at the..."
1,Azadirachtin,"For wheat crops, use when mites is seen at the..."
2,Trichoderma viride,"For maize crops, use when powdery mildew is se..."
3,Pseudomonas fluorescens,"For cotton crops, use when early blight is see..."
4,Bacillus subtilis,"For tomato crops, use when wilt is seen at the..."
...,...,...
605,Mancozeb,"For early fungal leaf spots and blight risk, u..."
606,Sulphur Dust,"For powdery mildew on vegetables, grapes, and ..."
607,Sticky Trap,"For monitoring and reducing whiteflies, aphids..."
608,Pheromone Trap,For monitoring fruit borer or stem borer activ...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Crop_Medicine     610 non-null    object
 1   Uses_When_to_Use  610 non-null    object
dtypes: object(2)
memory usage: 9.7+ KB


In [7]:
df.columns

Index(['Crop_Medicine', 'Uses_When_to_Use'], dtype='object')

In [8]:
df.index

RangeIndex(start=0, stop=610, step=1)

# RAG Evaluation

In [10]:
from langchain_community.document_loaders.csv_loader import CSVLoader

In [11]:
#intialize loder with csv path
loader = CSVLoader(file_path="./crop_medicine_uses_dataset.csv")
#load the data into documents
data = loader.load()

In [22]:
data

[Document(metadata={'source': './crop_medicine_uses_dataset.csv', 'row': 0}, page_content='Crop_Medicine: Neem Oil\nUses_When_to_Use: For rice crops, use when aphids is seen at the early crop stage. Use only as per label and local agriculture advice. Keep a record of field observations.'),
 Document(metadata={'source': './crop_medicine_uses_dataset.csv', 'row': 1}, page_content='Crop_Medicine: Azadirachtin\nUses_When_to_Use: For wheat crops, use when mites is seen at the during humid weather. Use only after confirming the problem in the field. Check weather before any field spray.'),
 Document(metadata={'source': './crop_medicine_uses_dataset.csv', 'row': 2}, page_content='Crop_Medicine: Trichoderma viride\nUses_When_to_Use: For maize crops, use when powdery mildew is seen at the before disease spreads widely. Use as a preventive support when risk is high. Prefer soil testing for repeated nutrient issues.'),
 Document(metadata={'source': './crop_medicine_uses_dataset.csv', 'row': 3}, p

In [23]:
from langchain_core.documents import Document

docs = []

for row in data:
    text = row.page_content

    # simple parsing
    lines = text.split("\n")
    
    medicine = lines[0].replace("Crop_Medicine: ", "")
    uses = lines[1].replace("Uses_When_to_Use: ", "")

    docs.append(
        Document(
            page_content=uses,
            metadata={
                "medicine": medicine,
                "source": row.metadata.get("source")
            }
        )
    )

In [16]:
#split into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 300,chunk_overlap = 50)

In [24]:
chunks = splitter.split_documents(docs)

In [38]:
chunks

[Document(metadata={'medicine': 'Neem Oil', 'source': './crop_medicine_uses_dataset.csv'}, page_content='For rice crops, use when aphids is seen at the early crop stage. Use only as per label and local agriculture advice. Keep a record of field observations.'),
 Document(metadata={'medicine': 'Azadirachtin', 'source': './crop_medicine_uses_dataset.csv'}, page_content='For wheat crops, use when mites is seen at the during humid weather. Use only after confirming the problem in the field. Check weather before any field spray.'),
 Document(metadata={'medicine': 'Trichoderma viride', 'source': './crop_medicine_uses_dataset.csv'}, page_content='For maize crops, use when powdery mildew is seen at the before disease spreads widely. Use as a preventive support when risk is high. Prefer soil testing for repeated nutrient issues.'),
 Document(metadata={'medicine': 'Pseudomonas fluorescens', 'source': './crop_medicine_uses_dataset.csv'}, page_content='For cotton crops, use when early blight is se

In [45]:
from langchain_ollama import OllamaEmbeddings,ChatOllama

In [46]:
llm = ChatOllama(model="mistral")

In [40]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [41]:
from langchain_community.vectorstores import FAISS

In [42]:
vectorStore = FAISS.from_documents(docs,embeddings)

In [44]:
#create a retriever
retriever = vectorStore.as_retriever(search_kwargs={"k": 2})

In [52]:
#. Define the RAG Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
#Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

parser = StrOutputParser()
#chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}

    | prompt
    | llm
    | parser
    
)

# Example Usage
print(chain.invoke("Give details about Neem Oil"))

 The provided context does not include any information about Neem Oil. Therefore, I cannot provide details about it based on this data.
